# Camada Gold — Data Quality Monitoring

Este notebook lê a tabela Delta `squad1.dq_monitoring_logs` e a Silver `squad1.silver_ecommerce_clientes` para gerar tabelas Gold de monitoramento de qualidade de dados.

Saídas geradas:

- `squad1.gold_dq_resumo_por_regra`: percentual de falha por regra por dia.
- `squad1.gold_dq_resumo_por_tabela`: percentual de registros limpos por tabela por hora.

As tabelas são gravadas como Delta no Databricks e também replicadas para o SQL Server Azure para consumo no Looker.

Estratégia anti-duplicidade: **full refresh idempotente**. As tabelas Gold são recalculadas a partir das fontes oficiais e gravadas com `overwrite`, evitando duplicidade mesmo quando o notebook é executado várias vezes ou por mais de uma pessoa.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

print("Iniciando Gold de Clientes — Data Quality")


## Parâmetros das tabelas

Execute o `config` antes deste notebook ou descomente o `%run` abaixo se o caminho estiver correto.


In [0]:
# Tabelas candidatas — ajuste aqui se seu ambiente estiver diferente
DQ_LOGS_CANDIDATAS = [
    "squad1.dq_monitoring_logs",
]

SILVER_CLIENTES_CANDIDATAS = [
    "squad1.silver.ecommerce_clientes",
    "squad1.silver.silver_ecommerce_clientes",
]

# Saídas Gold recomendadas no padrão catalog.schema.table
GOLD_SCHEMA = "squad1.gold"
GOLD_RESUMO_REGRA_TABLE = "squad1.gold.dq_clientes_resumo_por_regra"
GOLD_RESUMO_TABELA_TABLE = "squad1.gold.dq_clientes_resumo_por_tabela"
GOLD_REJEICOES_ARQUIVO_TABLE = "squad1.gold.dq_clientes_rejeicoes_por_arquivo"
GOLD_KPIS_GERAIS_TABLE = "squad1.gold.dq_clientes_kpis_gerais"

# Nome lógico usado nos logs da Silver
NOMES_TABELA_CLIENTES_LOGS = [
    "silver_ecommerce_clientes",
    "ecommerce_clientes",
    "clientes"
]

print("Gold schema:", GOLD_SCHEMA)


## Funções utilitárias

In [0]:
def tabela_existe(nome_tabela: str) -> bool:
    try:
        return spark.catalog.tableExists(nome_tabela)
    except Exception:
        try:
            spark.table(nome_tabela).limit(1).count()
            return True
        except Exception:
            return False


def primeira_tabela_existente(candidatas, descricao):
    for tabela in candidatas:
        if tabela_existe(tabela):
            print(f"{descricao} encontrada: {tabela}")
            return tabela
    raise Exception(
        f"Nenhuma tabela encontrada para {descricao}. Tabelas testadas: {candidatas}"
    )


def criar_schema_se_nao_existir(schema: str):
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
        print(f"Schema disponível: {schema}")
    except Exception as e:
        print(f"Aviso: não foi possível criar/verificar o schema {schema}: {e}")


def salvar_delta_overwrite(df, tabela: str):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela)
    )
    print(f"Tabela Gold Delta atualizada: {tabela}")


def coluna_existe(df, coluna: str) -> bool:
    return coluna in df.columns


## 2. Ler `dq_monitoring_logs` e Silver de clientes


In [0]:
DQ_LOGS_TABLE = primeira_tabela_existente(DQ_LOGS_CANDIDATAS, "dq_monitoring_logs")
SILVER_CLIENTES_TABLE = primeira_tabela_existente(SILVER_CLIENTES_CANDIDATAS, "Silver de clientes")

criar_schema_se_nao_existir(GOLD_SCHEMA)

df_logs = spark.table(DQ_LOGS_TABLE)
df_silver_clientes = spark.table(SILVER_CLIENTES_TABLE)

print("Registros em dq_monitoring_logs:", df_logs.count())
print("Registros na Silver de clientes:", df_silver_clientes.count())

display(df_logs.limit(10))
display(df_silver_clientes.limit(10))


## 3. Filtrar logs referentes à tabela de clientes

A Gold deve mostrar os erros gravados em `dq_monitoring_logs`, principalmente quantos dados inválidos foram detectados em cada regra.


In [0]:
# Normaliza nome da tabela no log para evitar diferença entre nomes físicos/lógicos
df_logs_clientes = (
    df_logs
    .withColumn("tabela_normalizada", F.lower(F.col("tabela")))
    .filter(
        F.col("tabela_normalizada").isin([x.lower() for x in NOMES_TABELA_CLIENTES_LOGS])
        | F.col("tabela_normalizada").contains("cliente")
    )
)

qtd_logs_clientes = df_logs_clientes.count()
print("Logs encontrados para clientes:", qtd_logs_clientes)

if qtd_logs_clientes == 0:
    print("ATENÇÃO: nenhum log de clientes encontrado em dq_monitoring_logs.")
    print("Verifique se a Silver de clientes gravou df_dq_monitoring_logs na tabela Delta de logs.")

# Apenas para visualização
if qtd_logs_clientes > 0:
    display(
        df_logs_clientes
        .select("run_id", "tabela", "regra", "status", "severidade", "qtd_registros_falhos", "qtd_registros_total", "timestamp_execucao", "arquivo_origem")
        .orderBy(F.desc("timestamp_execucao"))
        .limit(50)
    )


## 4. Gold — Quantidade de inválidos por regra por dia

Esta é a principal tabela para responder: **quantos dados inválidos foram detectados em cada regra?**


In [0]:
df_gold_resumo_por_regra = (
    df_logs_clientes
    .withColumn("data_referencia", F.to_date(F.col("timestamp_execucao")))
    .groupBy(
        "data_referencia",
        "tabela",
        "regra",
        "severidade"
    )
    .agg(
        F.sum(F.col("qtd_registros_falhos")).cast("int").alias("qtd_dados_invalidos"),
        F.sum(F.col("qtd_registros_total")).cast("int").alias("qtd_registros_avaliados"),
        F.countDistinct("arquivo_origem").cast("int").alias("qtd_arquivos_avaliados"),
        F.countDistinct("run_id").cast("int").alias("qtd_runs")
    )
    .withColumn(
        "percentual_falha",
        F.when(
            F.col("qtd_registros_avaliados") > 0,
            F.round((F.col("qtd_dados_invalidos") / F.col("qtd_registros_avaliados")) * 100, 4)
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "status_resumo",
        F.when(F.col("qtd_dados_invalidos") > 0, F.lit("FAIL")).otherwise(F.lit("PASS"))
    )
    .withColumn("gold_processed_at", F.current_timestamp())
    .orderBy("data_referencia", "regra")
)

display(df_gold_resumo_por_regra)



## 5. Gold — Rejeições por arquivo de origem

Mostra quais arquivos geraram mais falhas por regra.


In [0]:
df_gold_rejeicoes_por_arquivo = (
    df_logs_clientes
    .withColumn("data_referencia", F.to_date(F.col("timestamp_execucao")))
    .groupBy(
        "data_referencia",
        "arquivo_origem",
        "tabela",
        "regra",
        "severidade"
    )
    .agg(
        F.sum("qtd_registros_falhos").cast("int").alias("qtd_dados_invalidos"),
        F.sum("qtd_registros_total").cast("int").alias("qtd_registros_avaliados")
    )
    .withColumn(
        "percentual_falha",
        F.when(
            F.col("qtd_registros_avaliados") > 0,
            F.round((F.col("qtd_dados_invalidos") / F.col("qtd_registros_avaliados")) * 100, 4)
        ).otherwise(F.lit(0.0))
    )
    .withColumn("gold_processed_at", F.current_timestamp())
    .orderBy(F.desc("qtd_dados_invalidos"), "arquivo_origem", "regra")
)

display(df_gold_rejeicoes_por_arquivo)


## 6. Gold — Resumo por tabela por hora

Usa a Silver para calcular quantos registros ficaram limpos e inválidos por hora de processamento.


In [0]:
colunas_silver = df_silver_clientes.columns

if "silver_linha_valida" in colunas_silver:
    df_silver_base = df_silver_clientes.withColumn(
        "silver_linha_valida_calc",
        F.coalesce(F.col("silver_linha_valida").cast("boolean"), F.lit(False))
    )
else:
    flags_regras = [c for c in colunas_silver if c.startswith("r") and c.endswith("_falhou")]

    if len(flags_regras) == 0:
        raise Exception(
            "Não encontrei silver_linha_valida nem flags de regras terminando com _falhou na Silver de clientes."
        )

    condicao_falha = None
    for c in flags_regras:
        expr = F.coalesce(F.col(c).cast("boolean"), F.lit(False))
        condicao_falha = expr if condicao_falha is None else (condicao_falha | expr)

    df_silver_base = df_silver_clientes.withColumn(
        "silver_linha_valida_calc",
        ~condicao_falha
    )

if "silver_processed_at" in df_silver_base.columns:
    coluna_tempo_silver = "silver_processed_at"
elif "bronze_ingested_at" in df_silver_base.columns:
    coluna_tempo_silver = "bronze_ingested_at"
else:
    raise Exception("A Silver precisa ter silver_processed_at ou bronze_ingested_at para agregação por hora.")

print("Coluna de tempo usada:", coluna_tempo_silver)


In [0]:
df_gold_resumo_por_tabela = (
    df_silver_base
    .withColumn("hora_referencia", F.date_trunc("hour", F.col(coluna_tempo_silver)))
    .withColumn("tabela", F.lit("silver_ecommerce_clientes"))
    .groupBy("hora_referencia", "tabela")
    .agg(
        F.count("*").cast("int").alias("qtd_registros_total"),
        F.sum(F.when(F.col("silver_linha_valida_calc") == True, 1).otherwise(0)).cast("int").alias("qtd_registros_limpos"),
        F.sum(F.when(F.col("silver_linha_valida_calc") == False, 1).otherwise(0)).cast("int").alias("qtd_registros_invalidos")
    )
    .withColumn(
        "percentual_registros_limpos",
        F.when(
            F.col("qtd_registros_total") > 0,
            F.round((F.col("qtd_registros_limpos") / F.col("qtd_registros_total")) * 100, 4)
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "percentual_registros_invalidos",
        F.when(
            F.col("qtd_registros_total") > 0,
            F.round((F.col("qtd_registros_invalidos") / F.col("qtd_registros_total")) * 100, 4)
        ).otherwise(F.lit(0.0))
    )
    .withColumn("gold_processed_at", F.current_timestamp())
    .orderBy("hora_referencia")
)

display(df_gold_resumo_por_tabela)


## 7. Gold — KPIs gerais de qualidade dos clientes


In [0]:
# Agregados vindos dos logs
agg_logs = (
    df_logs_clientes
    .agg(
        F.sum("qtd_registros_falhos").cast("int").alias("total_falhas_regras"),
        F.countDistinct("regra").cast("int").alias("qtd_regras_avaliadas"),
        F.countDistinct("arquivo_origem").cast("int").alias("qtd_arquivos_avaliados"),
        F.countDistinct("run_id").cast("int").alias("qtd_runs")
    )
)

agg_silver = (
    df_silver_base
    .agg(
        F.count("*").cast("int").alias("total_registros_silver"),
        F.sum(F.when(F.col("silver_linha_valida_calc") == True, 1).otherwise(0)).cast("int").alias("total_registros_limpos"),
        F.sum(F.when(F.col("silver_linha_valida_calc") == False, 1).otherwise(0)).cast("int").alias("total_registros_invalidos")
    )
)

df_gold_kpis_gerais = (
    agg_silver
    .crossJoin(agg_logs)
    .withColumn(
        "percentual_registros_limpos",
        F.when(
            F.col("total_registros_silver") > 0,
            F.round((F.col("total_registros_limpos") / F.col("total_registros_silver")) * 100, 4)
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "percentual_registros_invalidos",
        F.when(
            F.col("total_registros_silver") > 0,
            F.round((F.col("total_registros_invalidos") / F.col("total_registros_silver")) * 100, 4)
        ).otherwise(F.lit(0.0))
    )
    .withColumn("tabela", F.lit("silver_ecommerce_clientes"))
    .withColumn("gold_processed_at", F.current_timestamp())
    .select(
        "tabela",
        "total_registros_silver",
        "total_registros_limpos",
        "total_registros_invalidos",
        "percentual_registros_limpos",
        "percentual_registros_invalidos",
        "total_falhas_regras",
        "qtd_regras_avaliadas",
        "qtd_arquivos_avaliados",
        "qtd_runs",
        "gold_processed_at"
    )
)

display(df_gold_kpis_gerais)


**Erros mais comuns**

In [0]:
df_gold_top_erros = (
    df_logs_clientes
    .groupBy("regra")
    .agg(
        F.sum("qtd_registros_falhos").alias("qtd_erros")
    )
    .orderBy(F.desc("qtd_erros"))
)

## Erros por severidade

In [0]:
df_gold_erros_por_severidade = (
    df_logs_clientes
    .groupBy("severidade")
    .agg(
        F.sum("qtd_registros_falhos").alias("qtd_erros")
    )
)

## Ranking de regras

In [0]:
from pyspark.sql.window import Window

w = Window.orderBy(F.desc("qtd_erros"))

df_gold_ranking_regras = (
    df_gold_top_erros
    .withColumn(
        "ranking",
        F.row_number().over(w)
    )
)

## 8. Gravar tabelas Gold em Delta

As tabelas são recriadas com `overwrite`, evitando duplicidade a cada execução.


In [0]:
salvar_delta_overwrite(df_gold_resumo_por_regra, GOLD_RESUMO_REGRA_TABLE)
salvar_delta_overwrite(df_gold_resumo_por_tabela, GOLD_RESUMO_TABELA_TABLE)
salvar_delta_overwrite(df_gold_rejeicoes_por_arquivo, GOLD_REJEICOES_ARQUIVO_TABLE)
salvar_delta_overwrite(df_gold_kpis_gerais, GOLD_KPIS_GERAIS_TABLE)
salvar_delta_overwrite(df_gold_ranking_regras, GOLD_RANKING_REGRAS)
salvar_delta_overwrite(df_gold_erros_por_severidade, GOLD_ERROS_POR_SEVERIDADE)
salvar_delta_overwrite(df_gold_top_erros, GOLD_TOP_ERROS)


## 9. Replica opcional para SQL Server Azure

A réplica é executada apenas se as variáveis JDBC estiverem carregadas no notebook (`JDBC_HOSTNAME`, `JDBC_DATABASE`, `JDBC_USERNAME`, `JDBC_PASSWORD`).


In [0]:
def jdbc_config_disponivel():
    variaveis = ["JDBC_HOSTNAME", "JDBC_DATABASE", "JDBC_USERNAME", "JDBC_PASSWORD"]
    return all(v in globals() and globals()[v] not in [None, ""] for v in variaveis)


def gravar_sqlserver_gold(df, schema: str, tabela: str):
    tabela_destino = f"{schema}.{tabela}"
    (
        df.write
        .format("sqlserver")
        .mode("overwrite")
        .option("host", JDBC_HOSTNAME)
        .option("port", "1433")
        .option("database", JDBC_DATABASE)
        .option("user", JDBC_USERNAME)
        .option("password", JDBC_PASSWORD)
        .option("dbtable", tabela_destino)
        .option("encrypt", "true")
        .option("trustServerCertificate", "false")
        .save()
    )
    print(f"Tabela SQL Server atualizada: {tabela_destino}")

if jdbc_config_disponivel():
    SQL_SCHEMA = "squad1"
    gravar_sqlserver_gold(df_gold_resumo_por_regra, SQL_SCHEMA, "gold_clientes_dq_resumo_por_regra")
    gravar_sqlserver_gold(df_gold_resumo_por_tabela, SQL_SCHEMA, "gold_clientes_dq_resumo_por_tabela")
    gravar_sqlserver_gold(df_gold_rejeicoes_por_arquivo, SQL_SCHEMA, "gold_clientes_dq_rejeicoes_por_arquivo")
    gravar_sqlserver_gold(df_gold_kpis_gerais, SQL_SCHEMA, "gold_clientes_dq_kpis_gerais")
else:
    print("Variáveis JDBC não encontradas. Réplica para SQL Server ignorada.")
    print("As tabelas Gold Delta foram criadas normalmente.")


## 10. Validação final


In [0]:
print("=" * 80)
print("GOLD DE CLIENTES CONCLUÍDA")
print("=" * 80)

for tabela in [
    GOLD_RESUMO_REGRA_TABLE,
    GOLD_RESUMO_TABELA_TABLE,
    GOLD_REJEICOES_ARQUIVO_TABLE,
    GOLD_KPIS_GERAIS_TABLE
]:
    if spark.catalog.tableExists(tabela):
        qtd = spark.table(tabela).count()
        print(f"{tabela}: {qtd} registros")
    else:
        print(f"{tabela}: NÃO EXISTE")

print("
Resumo por regra:")
display(spark.table(GOLD_RESUMO_REGRA_TABLE).orderBy(F.desc("qtd_dados_invalidos")))

print("
KPIs gerais:")
display(spark.table(GOLD_KPIS_GERAIS_TABLE))
